In [7]:
import json
import re

In [8]:

# ==========================================
# 1. CÁC HÀM TIỆN ÍCH (HELPER FUNCTIONS)
# ==========================================

def get_full_subtree_indices(head_idx, dependencies):
    """Đệ quy tìm toàn bộ các token thuộc về một nhánh (con, cháu)."""
    indices = set([head_idx])
    added = True
    while added:
        added = False
        for token in dependencies:
            if token['head_idx'] in indices and token['token_idx'] not in indices:
                indices.add(token['token_idx'])
                added = True
    return indices

def get_subtree_phrase_with_exclusion(token_idx, dependencies, excluded_head_indices=None):
    """Trích xuất cụm từ, chặn đệ quy sâu bằng Blacklist để chống Overlapping."""
    if excluded_head_indices is None:
        excluded_head_indices = set()
        
    blacklisted_indices = set()
    for ex_idx in excluded_head_indices:
        blacklisted_indices.update(get_full_subtree_indices(ex_idx, dependencies))
        
    subtree_indices = set([token_idx])
    added = True
    while added:
        added = False
        for token in dependencies:
            if token['token_idx'] in blacklisted_indices:
                continue
            if token['head_idx'] in subtree_indices and token['token_idx'] not in subtree_indices:
                subtree_indices.add(token['token_idx'])
                added = True
                
    sorted_indices = sorted(list(subtree_indices))
    phrase = [token['token'] for idx in sorted_indices for token in dependencies if token['token_idx'] == idx]
    return " ".join(phrase)

def clean_punctuation(text):
    """Dọn dẹp dấu câu thừa ở 2 đầu."""
    if not text:
        return text
    cleaned = re.sub(r'^[\s,:\.\-]+|[\s,:\.\-]+$', '', text).strip()
    return cleaned

# Các hàm kiểm tra ngữ nghĩa hẹp (Tránh lỗi "nuốt" chữ)
def is_time_node(token_idx, deps, time_keywords):
    current_token = next((t for t in deps if t['token_idx'] == token_idx), None)
    if not current_token: return False
    
    if any(kw in current_token['token'].lower() for kw in time_keywords):
        return True
        
    # Chỉ kiểm tra con trực tiếp là giới từ (vào, lúc, trong vòng)
    for t in deps:
        if t['head_idx'] == token_idx and t['dep'].lower() in ['case', 'prep', 'mark']:
            if any(kw in t['token'].lower() for kw in time_keywords):
                return True
    return False

def is_recipient_node(token_idx, deps, recipient_keywords):
    for t in deps:
        if t['head_idx'] == token_idx and t['dep'].lower() in ['case', 'prep', 'mark']:
            if any(t['token'].lower() == kw for kw in recipient_keywords):
                return True
    return False


In [9]:

# ==========================================
# 2. HÀM TRÍCH XUẤT CHÍNH
# ==========================================

def extract_srl_hybrid(dependency_file, ner_file, output_file):
    # Load Data
    with open(dependency_file, 'r', encoding='utf-8') as f:
        deps_data = json.load(f)
    with open(ner_file, 'r', encoding='utf-8') as f:
        ner_data = json.load(f)

    # Tạo từ điển NER (Key: Text -> Value: Entities) để map nhanh
    ner_dict = {item['text']: item['entities'] for item in ner_data}

    srl_results = []
    
    recipient_keywords = ["cho", "tới", "đến"]
    time_keywords = ["trước", "sau", "vào", "lúc", "ngày", "tháng", "năm", "hạn", "trong vòng"]
    condition_keywords = ["nếu", "khi", "trong trường hợp", "trừ khi", "chậm tiến độ", "vi phạm"]

    for clause in deps_data:
        clause_id = clause.get('clause_id')
        text = clause.get('text')
        deps = clause.get('dependencies', [])
        entities = ner_dict.get(text, []) # Lấy NER cho câu hiện tại
        
        roles = {"Agent": "", "Predicate": "", "Theme": "", "Recipient": "", "Time": "", "Condition": ""}
        role_heads = {"Agent": None, "Theme": None, "Recipient": None, "Time": None, "Condition": None}
        
        root_token = None
        for token in deps:
            if token['dep'] in ['root', 'ROOT']:
                root_token = token
                break
                
        if root_token:
            predicate_tokens = [root_token]
            action_center_indices = [root_token['token_idx']]
            
            # Gộp Cụm Động từ / Bị động / Phủ định
            if root_token['token'].lower() in ['được', 'bị', 'phải', 'không', 'chưa', 'cam kết', 'đồng ý']:
                for token in deps:
                    if token['head_idx'] == root_token['token_idx'] and token['dep'].lower() in ['vmod', 'xcomp', 'ccomp']:
                        predicate_tokens.append(token)
                        action_center_indices.append(token['token_idx'])
            
            # Xử lý ngoại lệ Câu Định Nghĩa (VD: "Phạm vi :", "Địa điểm giao hàng :")
            # Nếu cụm ROOT nối trực tiếp với dấu ":"
            has_colon = any(t['token'] == ':' and t['head_idx'] == root_token['token_idx'] for t in deps)
            if has_colon or root_token['token'].lower() in ['phạm vi', 'địa điểm', 'tổng thu nhập', 'loại hợp đồng']:
                # Biến nguyên cụm danh từ ROOT thành Predicate
                pred_phrase = get_subtree_phrase_with_exclusion(root_token['token_idx'], deps)
                roles["Predicate"] = clean_punctuation(pred_phrase.split(':')[0]) # Lấy phần trước dấu hai chấm
                
                # Theme sẽ là phần còn lại sau dấu hai chấm
                for token in deps:
                    if token['head_idx'] == root_token['token_idx'] and token['token'] != ':':
                        role_heads["Theme"] = token['token_idx']
                        break
            # ... [Phần code phía trên giữ nguyên] ...
            else:
                predicate_tokens.sort(key=lambda x: x['token_idx'])
                roles["Predicate"] = " ".join([t['token'] for t in predicate_tokens])

                # Xác định xem Predicate có phải Động từ Mệnh lệnh/Cầu khiến không
                directive_verbs = ['nghiêm cấm', 'cấm', 'yêu cầu', 'đề nghị', 'bắt buộc', 'buộc']
                is_directive = any(v in roles["Predicate"].lower() for v in directive_verbs)

                # Phân bổ các Nhãn vai trò (Roles)
                for token in deps:
                    if token['head_idx'] in action_center_indices:
                        dep_label = token['dep'].lower()
                        temp_phrase = get_subtree_phrase_with_exclusion(token['token_idx'], deps).lower()

                        if dep_label in ['sub', 'nsubj'] and not role_heads["Agent"]:
                            role_heads["Agent"] = token['token_idx']
                            
                        elif dep_label in ['advcl'] or any(temp_phrase.startswith(kw) for kw in condition_keywords):
                            role_heads["Condition"] = token['token_idx']
                            
                        elif is_time_node(token['token_idx'], deps, time_keywords):
                            role_heads["Time"] = token['token_idx']
                                
                        elif dep_label in ['iobj'] or is_recipient_node(token['token_idx'], deps, recipient_keywords):
                            role_heads["Recipient"] = token['token_idx']
                            
                        elif dep_label in ['dob', 'obj', 'nmod']:
                            # NẾU là từ Mệnh lệnh (Nghiêm cấm Bên A...), Tân ngữ là Agent
                            if is_directive and not role_heads["Agent"]:
                                role_heads["Agent"] = token['token_idx']
                            # Nếu bình thường, Tân ngữ là Theme
                            elif not role_heads["Theme"] and token['token_idx'] not in action_center_indices:
                                role_heads["Theme"] = token['token_idx']
                                
                        elif dep_label in ['xcomp', 'ccomp', 'vmod']:
                            # Hành động bị cấm / được yêu cầu (truy cập, sao chép...) sẽ là Theme
                            if is_directive and not role_heads["Theme"]:
                                role_heads["Theme"] = token['token_idx']
                            elif not role_heads["Theme"] and token['token_idx'] not in action_center_indices:
                                role_heads["Theme"] = token['token_idx']

            # Backup Theme nếu còn sót
            if not role_heads["Theme"]:
                for token in deps:
                    if token['head_idx'] in action_center_indices and token['token_idx'] not in action_center_indices:
                        if token['token_idx'] not in role_heads.values():
                            role_heads["Theme"] = token['token_idx']
                            break
            # ... [Phần code trích xuất chuỗi và NER phía dưới giữ nguyên] ...

            # TRÍCH XUẤT CHUỖI & BLACKLIST
            blacklist_for_theme = set()
            for r, idx in role_heads.items():
                if r not in ["Agent", "Theme"] and idx is not None:
                    blacklist_for_theme.add(idx)

            for r, idx in role_heads.items():
                if idx is not None:
                    if r == "Theme":
                        raw_phrase = get_subtree_phrase_with_exclusion(idx, deps, excluded_head_indices=blacklist_for_theme)
                    else:
                        raw_phrase = get_subtree_phrase_with_exclusion(idx, deps)
                        
                    # Lọc sạch từ nối thừa
                    if r == "Recipient":
                        for kw in recipient_keywords:
                            if raw_phrase.lower().startswith(kw + " "):
                                raw_phrase = raw_phrase[len(kw)+1:].strip()
                                break
                    
                    roles[r] = clean_punctuation(raw_phrase)

        roles["Predicate"] = clean_punctuation(roles["Predicate"])

        srl_results.append({
            "clause_id": clause_id,
            "text": text,
            "predicate": roles["Predicate"],
            "roles": roles
        })

    # ==========================================
    # 3. HẬU XỬ LÝ (POST-PROCESSING VỚI NER)
    # ==========================================
    for item in srl_results:
        text = item["text"]
        roles = item["roles"]
        entities = ner_dict.get(text, [])
        
        # Sửa lỗi Overlap hiển nhiên (ví dụ: Theme lấn át Recipient)
        if roles["Recipient"] and roles["Recipient"] in roles["Theme"]:
            roles["Theme"] = clean_punctuation(roles["Theme"].replace(roles["Recipient"], "").replace("cho", "").strip())
            
        # Dùng NER Date để validate lại Time nếu Time đang bị rỗng
        if not roles["Time"]:
            for ent in entities:
                if ent["label"] == "DATE":
                    # Tìm chuỗi chứa DATE để gán làm Time (heuristic nhẹ)
                    if ent["text"] in roles["Theme"]:
                         roles["Theme"] = roles["Theme"].replace(ent["text"], "").strip()
                         roles["Time"] = clean_punctuation(ent["text"])

    # Ghi kết quả
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(srl_results, f, ensure_ascii=False, indent=4)
        
    print(f"✅ Đã xử lý xong {len(srl_results)} mệnh đề. Kết quả lưu tại: {output_file}")


# Chạy chương trình
extract_srl_hybrid(
    dependency_file='output/dependency.json', 
    ner_file='output/ner_results.json', 
    output_file='output/2.2_predicted_srl.json'
)

✅ Đã xử lý xong 166 mệnh đề. Kết quả lưu tại: output/2.2_predicted_srl.json
